In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import warnings; warnings.filterwarnings('ignore')
import numpy as np
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
DATA = '../data'
labels = np.load(f'{DATA}/labels.npy')
sets = {
    'Frozen tiny':   np.load(f'{DATA}/brainomni_features_tiny.npy'),
    'Frozen base':   np.load(f'{DATA}/brainomni_features_base.npy'),
    'Handcrafted':   np.load(f'{DATA}/baseline_features.npy'),
}
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
print({k: v.shape for k, v in sets.items()})

In [ ]:
# Two classifiers, both standardized:
#  - L2 LR  : plain logistic regression (your current baseline)
#  - Elastic: logistic regression with elastic-net penalty; C and l1_ratio tuned by an
#             INNER CV, wrapped in the outer 5-fold, so tuning never sees the test fold (nested CV).
def plain_lr():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))

def elastic():
    return make_pipeline(StandardScaler(),
        LogisticRegressionCV(penalty='elasticnet', solver='saga',
            l1_ratios=[0.2, 0.5, 0.8], Cs=8, max_iter=3000, cv=3,
            scoring='accuracy', n_jobs=-1))

print(f'{"feature set":14s} {"model":11s} {"acc":>6} {"AUC":>7}')
results = {}
for name, X in sets.items():
    for mname, mk in [('L2 LR', plain_lr), ('ElasticNet', elastic)]:
        r = cross_validate(mk(), X, labels, cv=outer, scoring=['accuracy', 'roc_auc'])
        acc = r['test_accuracy'].mean(); auc = r['test_roc_auc'].mean()
        results[(name, mname)] = (acc, auc)
        print(f'{name:14s} {mname:11s} {acc:.3f}  {auc:.3f}', flush=True)

In [ ]:
# Summary: does elastic net beat plain L2 LR on each feature set?
print('Elastic net vs L2 LR (accuracy):')
for name in ['Frozen tiny', 'Frozen base', 'Handcrafted']:
    a_l2 = results[(name, 'L2 LR')][0]; a_en = results[(name, 'ElasticNet')][0]
    d = a_en - a_l2
    verdict = 'elastic net higher' if d > 0.005 else ('about the same' if abs(d) <= 0.005 else 'L2 higher')
    print(f'  {name:14s}: L2 {a_l2:.3f}  ->  Elastic {a_en:.3f}   ({d:+.3f}, {verdict})')